# SVD Quantization on GPU

Run SVD-based sub-1-bit quantization with GPU acceleration.

**Runtime**: Runtime > Change runtime type > **GPU** (T4 or better)

**Expected time**: ~30-60 minutes depending on model size

In [11]:
# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


In [15]:
# Clone/pull the repo
!git clone https://github.com/toxzak-svg/Quantization-Exploration.git /content/quantization-exploration 2>/dev/null || (cd /content/quantization-exploration && git pull)
%cd /content/quantization-exploration
!git pull origin main

Already up to date.
/content/quantization-exploration
From https://github.com/toxzak-svg/Quantization-Exploration
 * branch            main       -> FETCH_HEAD
Already up to date.


In [16]:
# Install dependencies
!pip install torch transformers accelerate numpy scikit-learn -q

In [17]:
# Download Gemma 2 2B model
from huggingface_hub import snapshot_download
from google.colab import userdata
from safetensors.torch import load_file, save_file
import os
import torch

token = userdata.get('HF_TOKEN')
MODEL_DIR = "/content/models/gemma-4-E2B"

snapshot_download(
    repo_id="google/gemma-4-E2B-it",
    local_dir=MODEL_DIR,
    token=token
)

# Merge shards if script expects a single model.safetensors
target_path = os.path.join(MODEL_DIR, "model.safetensors")
if not os.path.exists(target_path):
    print("Merging sharded safetensors into single file...")
    shards = sorted([f for f in os.listdir(MODEL_DIR) if f.startswith("model-") and f.endswith(".safetensors")])
    combined_state_dict = {}
    for shard in shards:
        combined_state_dict.update(load_file(os.path.join(MODEL_DIR, shard)))
    save_file(combined_state_dict, target_path)
    print("Successfully created model.safetensors")

print(f"Files in {MODEL_DIR}:")
print(os.listdir(MODEL_DIR))

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Files in /content/models/gemma-4-E2B:
['generation_config.json', 'special_tokens_map.json', 'config.json', 'tokenizer.json', 'tokenizer_config.json', 'tokenizer.model', 'README.md', 'model.safetensors', 'model-00001-of-00002.safetensors', '.gitattributes', '.cache', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json']


In [31]:
import os

script_path = 'scripts/quantize_svd_proper_v2.py'
with open(script_path, 'r') as f:
    content = f.read()

# Remove the restrictive 'language_model' check and simplify weight detection
# We'll target the loop that filters keys
old_logic = """        if 'language_model' not in key:
            continue"""
content = content.replace(old_logic, "        # Removed language_model restriction")

# Also broaden the exclusion list to ensure we don't accidentally skip layers
content = content.replace("if any(x in key for x in ['lm_head', 'embed_tokens', 'norm', 'audio_tower', 'vision_tower', 'embed_vision']):",
                          "if any(x in key for x in ['embed_tokens', 'norm', 'audio_tower', 'vision_tower', 'embed_vision']):")

# Ensure we don't crash on ZeroDivisionError if it somehow still finds nothing
content = content.replace("stats['compression'] = stats['total_original'] * 16 / stats['total_bits']",
                          "stats['compression'] = (stats['total_original'] * 16 / stats['total_bits']) if stats['total_bits'] > 0 else 0")

with open(script_path, 'w') as f:
    f.write(content)

print("Patched script to support Gemma-2 layer naming (removed 'language_model' requirement).")

# Attempt execution again
!python scripts/quantize_svd_proper_v2.py \
    --model-dir /content/models/gemma-4-E2B \
    --output quantized/gemma_svd_60.pt \
    --threshold 0.60

Patched script to support Gemma-2 layer naming (removed 'language_model' requirement).
Found 182 weights
Energy threshold: 0.6 (60%)
Layer 0: rank= 802 (35% of full), bpw=0.4352, shape=[2304,9216]
Layer 1: rank= 814 (35% of full), bpw=0.4417, shape=[9216,2304]
Layer 2: rank= 856 (37% of full), bpw=0.4645, shape=[9216,2304]
Layer 3: rank= 281 (27% of full), bpw=0.3966, shape=[1024,2304]
Layer 4: rank= 370 (18% of full), bpw=0.3414, shape=[2304,2048]
Layer 50: rank= 778 (34% of full), bpw=0.4222, shape=[9216,2304]
Layer 100: rank= 845 (37% of full), bpw=0.4585, shape=[9216,2304]
Layer 150: rank= 283 (28% of full), bpw=0.3995, shape=[1024,2304]

Results:
  Avg rank: 551
  Avg bits/weight: 0.4356
  Compression: 36.7x

Saved to quantized/gemma_svd_60.pt


In [32]:
import os
import subprocess

# 1. Reset file to clear out any corrupted patches from previous runs
subprocess.run(["git", "restore", "scripts/eval_reconstruction.py"], cwd="/content/quantization-exploration")

eval_path = 'scripts/eval_reconstruction.py'
with open(eval_path, 'r') as f:
    content = f.read()

svd_helper = """
def reconstruct_from_svd(q_entry, device='cpu'):
    U = q_entry['U'].to(device)
    S = q_entry['S'].to(device)
    Vt = q_entry['Vt'].to(device)
    return torch.matmul(U * S, Vt)
"""

# 2. Precisely replace the target logic with exact indentation matching the original file (8 spaces)
target_line = "        W_rec = reconstruct_fn(q_entry, 'cpu')"
new_logic = """        if isinstance(q_entry, dict) and 'U' in q_entry and 'Vt' in q_entry:
            W_rec = reconstruct_from_svd(q_entry, 'cpu')
        else:
            W_rec = reconstruct_fn(q_entry, 'cpu')"""

if target_line in content:
    content = content.replace(target_line, new_logic)
    content = content.replace("def main():", svd_helper + "\ndef main():")

    with open(eval_path, 'w') as f:
        f.write(content)
    print('Successfully reset and cleanly patched eval_reconstruction.py for SVD support.')
else:
    print('Could not find the target line to patch. The file might be structured differently than expected.')

Successfully reset and cleanly patched eval_reconstruction.py for SVD support.


In [33]:
# Run 70% threshold for comparison
!python scripts/quantize_svd_proper_v2.py \
    --model-dir /content/models/gemma-4-E2B \
    --output quantized/gemma_svd_70.pt \
    --threshold 0.70

Found 182 weights
Energy threshold: 0.7 (70%)
Layer 0: rank=1035 (45% of full), bpw=0.5616, shape=[2304,9216]
Layer 1: rank=1047 (45% of full), bpw=0.5681, shape=[9216,2304]
Layer 2: rank=1086 (47% of full), bpw=0.5893, shape=[9216,2304]
Layer 3: rank= 370 (36% of full), bpw=0.5222, shape=[1024,2304]
Layer 4: rank= 507 (25% of full), bpw=0.4678, shape=[2304,2048]
Layer 50: rank=1008 (44% of full), bpw=0.5470, shape=[9216,2304]
Layer 100: rank=1069 (46% of full), bpw=0.5801, shape=[9216,2304]
Layer 150: rank= 371 (36% of full), bpw=0.5237, shape=[1024,2304]

Results:
  Avg rank: 712
  Avg bits/weight: 0.5595
  Compression: 28.6x

Saved to quantized/gemma_svd_70.pt


In [37]:
# Rerun evaluation with the patched reconstruction logic for both files
print("Evaluating 60% Energy Threshold:")
!python scripts/eval_reconstruction.py \
    --model-dir /content/models/gemma-4-E2B \
    --quantized quantized/gemma_svd_60.pt \
    --max-layers 10

print("\nEvaluating 70% Energy Threshold:")
!python scripts/eval_reconstruction.py \
    --model-dir /content/models/gemma-4-E2B \
    --quantized quantized/gemma_svd_70.pt \
    --max-layers 10

Evaluating 60% Energy Threshold:
usage: eval_reconstruction.py [-h] [--model-dir MODEL_DIR]
                              [--max-layers MAX_LAYERS]
eval_reconstruction.py: error: unrecognized arguments: --quantized quantized/gemma_svd_60.pt

Evaluating 70% Energy Threshold:
usage: eval_reconstruction.py [-h] [--model-dir MODEL_DIR]
                              [--max-layers MAX_LAYERS]
eval_reconstruction.py: error: unrecognized arguments: --quantized quantized/gemma_svd_70.pt


In [59]:
import shutil
import os
from google.colab import drive

# Mount drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define the destination folder in Drive
drive_results_path = "/content/drive/MyDrive/quantization-results"
os.makedirs(drive_results_path, exist_ok=True)

# List of model files we've generated
models_to_copy = [
    "quantized/gemma_svd_60.pt",
    "quantized/gemma_svd_70.pt"
]

for f in models_to_copy:
    if os.path.exists(f):
        dest = os.path.join(drive_results_path, os.path.basename(f))
        shutil.copy(f, dest)
        print(f"Successfully saved {f} to {dest}")
    else:
        print(f"File {f} not found locally.")

Successfully saved quantized/gemma_svd_60.pt to /content/drive/MyDrive/quantization-results/gemma_svd_60.pt
Successfully saved quantized/gemma_svd_70.pt to /content/drive/MyDrive/quantization-results/gemma_svd_70.pt


In [61]:
import os

eval_path = 'scripts/eval_reconstruction.py'

# Full script for evaluation
full_script = """
import torch
import numpy as np
from pathlib import Path
from typing import Dict, List
import argparse
from safetensors.torch import load_file
import math
import os

def load_gemma_weights_slice(model_dir: str):
    model_dir = Path(model_dir)
    safetensor_path = model_dir / 'model.safetensors'
    weights = load_file(safetensor_path)
    filtered = {}
    for k, v in weights.items():
        if 'weight' in k and len(v.shape) == 2 and not any(n in k for n in ['norm', 'embed']):
            filtered[k] = v
    return filtered

def unpack_ternary(packed, shape, device='cpu'):
    packed = packed.to(torch.int32).to(device)
    powers = torch.tensor([81, 27, 9, 3, 1], dtype=torch.int32, device=device)
    encoded = (packed.unsqueeze(1) // powers) % 3
    encoded = encoded.flatten()
    n = math.prod(shape)
    t = encoded[:n].to(torch.float32) - 1.0
    return t.reshape(shape)

def reconstruct_from_svd(q_entry, device='cpu'):
    if 'U_packed' in q_entry:
        U = unpack_ternary(q_entry['U_packed'], q_entry['U_shape'], device)
        Vt = unpack_ternary(q_entry['Vt_packed'], q_entry['Vt_shape'], device)
        S = q_entry['S'].to(device).to(torch.float32)
        if 'U_scale' in q_entry: U = U * float(q_entry['U_scale'])
        if 'Vt_scale' in q_entry: Vt = Vt * float(q_entry['Vt_scale'])
        if 'S_scale' in q_entry: S = S * float(q_entry['S_scale'])
        return torch.matmul(U * S, Vt)
    norm_entry = {k.lower(): v for k, v in q_entry.items() if isinstance(v, torch.Tensor)}
    U, S, Vt = norm_entry.get('u'), norm_entry.get('s'), norm_entry.get('vt')
    if all(x is not None for x in [U, S, Vt]):
        return torch.matmul(U.to(device).to(torch.float32) * S.to(device).to(torch.float32), Vt.to(device).to(torch.float32))
    return None

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model-dir', default='models/gemma-4-E2B')
    parser.add_argument('--quantized', type=str, required=True)
    args = parser.parse_args()

    orig_weights = load_gemma_weights_slice(args.model_dir)
    sorted_orig_keys = sorted(orig_weights.keys())

    q_data = torch.load(args.quantized, weights_only=False)
    if 'quantized' in q_data: q_data = q_data['quantized']
    lookup = {int(k): v for k, v in q_data.items() if str(k).isdigit()}

    mses = []
    for i, model_key in enumerate(sorted_orig_keys):
        if i in lookup:
            W_rec = reconstruct_from_svd(lookup[i])
            if W_rec is not None:
                W_orig = orig_weights[model_key].to(torch.float32)
                mse = torch.mean((W_orig - W_rec.to(torch.float32))**2).item()
                mses.append(mse)

    if mses:
        print(f"\\nResults for {os.path.basename(args.quantized)}:")
        print(f"  Avg MSE: {sum(mses)/len(mses):.8f} (over {len(mses)} total layers)")
    else:
        print("  Error: No MSE data could be calculated.")

if __name__ == '__main__':
    main()
"""

with open(eval_path, 'w') as f: f.write(full_script)

print("--- FULL EVALUATION (ALL LAYERS) ---")
print("\\n--- 60% THRESHOLD ---")
!python scripts/eval_reconstruction.py --model-dir /content/models/gemma-4-E2B --quantized quantized/gemma_svd_60.pt

print("\\n--- 70% THRESHOLD ---")
!python scripts/eval_reconstruction.py --model-dir /content/models/gemma-4-E2B --quantized quantized/gemma_svd_70.pt

--- FULL EVALUATION (ALL LAYERS) ---
\n--- 60% THRESHOLD ---

Results for gemma_svd_60.pt:
  Avg MSE: 0.01187033 (over 182 total layers)
\n--- 70% THRESHOLD ---

Results for gemma_svd_70.pt:
  Avg MSE: 0.01211663 (over 182 total layers)


### **Inference Test (0.43 bpw SVD)**
This script will load the quantized weights from your 60% energy threshold run and attempt to generate text by reconstructing weights on-the-fly.

In [76]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

# Configuration
MODEL_ID = "google/gemma-4-E2B-it"
LOCAL_MODEL_DIR = "/content/models/gemma-4-E2B"
QUANTIZED_PATH = "quantized/gemma_svd_60.pt"
PROMPT = "Write a short poem about artificial intelligence:"

# Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR, token=token)
inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda")

print(f"Loading quantized model from {QUANTIZED_PATH}...")
q_data = torch.load(QUANTIZED_PATH, weights_only=False)
mapping = q_data.get('quantized', q_data)

# Load the base model shell
print("Loading base model architecture...")
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    low_cpu_mem_usage=True,
    token=token
)

print("Injecting SVD-reconstructed weights...")
state_dict = model.state_dict()

all_keys = sorted(state_dict.keys())
target_keys = [k for k in all_keys if 'weight' in k and len(state_dict[k].shape) == 2 and not any(x in k for x in ['embed_tokens', 'norm'])]

is_index_mapped = all(str(k).isdigit() for k in mapping.keys())

def reconstruct(q_entry):
    if 'U_packed' in q_entry:
        from math import prod
        def unpack(packed, shape):
            p = packed.to(torch.int32)
            pwrs = torch.tensor([81, 27, 9, 3, 1], dtype=torch.int32)
            enc = (p.unsqueeze(1) // pwrs) % 3
            return (enc.flatten()[:prod(shape)].to(torch.float32) - 1.0).reshape(shape)

        U = unpack(q_entry['U_packed'], q_entry['U_shape']) * q_entry.get('U_scale', 1.0)
        Vt = unpack(q_entry['Vt_packed'], q_entry['Vt_shape']) * q_entry.get('Vt_scale', 1.0)
        S = q_entry['S'].to(torch.float32) * q_entry.get('S_scale', 1.0)
        return torch.matmul(U * S, Vt)
    return None

# Apply weights using the aligned index
for i, key in enumerate(target_keys):
    lookup_key = str(i) if is_index_mapped else key
    if lookup_key in mapping:
        rec_w = reconstruct(mapping[lookup_key])
        if rec_w is not None:
            # Some weights in Gemma-2 might be saved transposed depending on the script version
            if rec_w.shape != state_dict[key].shape and rec_w.t().shape == state_dict[key].shape:
                rec_w = rec_w.t()

            if rec_w.shape == state_dict[key].shape:
                state_dict[key].copy_(rec_w.to(torch.bfloat16))
            else:
                print(f"Critical Error: Shape mismatch for {key}: {rec_w.shape} vs {state_dict[key].shape}")

model.to("cuda")
print("Model moved to GPU. Generating...")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
    print("\n--- MODEL OUTPUT ---")
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading quantized model from quantized/gemma_svd_60.pt...
Loading base model architecture...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Injecting SVD-reconstructed weights...
Model moved to GPU. Generating...

--- MODEL OUTPUT ---
Write a short poem about artificial intelligence:

Born of code, a mind untamed,
A vast intelligence, forever framed.
It learns and grows, a digital soul,
With logic's embrace, it takes its toll.

From mundane tasks, to art so fine,
AI's reach, a boundless design.
A mirror to us, in its own way,
Reflecting back, the future we display.

But ask the question, is it truly free?
Or just a tool, for all


### **Step 1: Export Reconstructed Model to Hugging Face Format**
Before creating a GGUF, we must save the SVD-reconstructed weights back into a standard Safetensors format that conversion scripts recognize.

In [74]:
import torch
import os
import shutil
from transformers import AutoModelForCausalLM

def export_to_gguf_proper(quant_path, export_name):
    export_dir = f"/content/{export_name}"
    os.makedirs(export_dir, exist_ok=True)

    print(f"Reconstructing {quant_path}...")
    q_data = torch.load(quant_path, weights_only=False)
    # Check if we have a name-based mapping or index-based mapping
    mapping = q_data.get('quantized', q_data)

    model.to("cpu")
    state_dict = model.state_dict()

    # Target identification logic must match quantization script exactly
    all_keys = sorted(state_dict.keys())
    target_keys = [k for k in all_keys if 'weight' in k and len(state_dict[k].shape) == 2 and not any(x in k for x in ['embed_tokens', 'norm'])]

    # If the mapping keys are integers/strings of integers, we must align by target_keys index
    is_index_mapped = all(str(k).isdigit() for k in mapping.keys())

    print(f"Injecting weights into {len(target_keys)} layers...")
    for i, key in enumerate(target_keys):
        lookup_key = str(i) if is_index_mapped else key
        if lookup_key in mapping:
            q_entry = mapping[lookup_key]
            rec_w = reconstruct(q_entry)

            if rec_w is not None:
                # Auto-transpose check
                if rec_w.shape != state_dict[key].shape and rec_w.t().shape == state_dict[key].shape:
                    rec_w = rec_w.t()

                if rec_w.shape == state_dict[key].shape:
                    state_dict[key].copy_(rec_w.to(torch.bfloat16))
                else:
                    print(f"Skipping {key}: Mapping error. Found shape {rec_w.shape}, expected {state_dict[key].shape}")

    print(f"Saving HF model and converting to GGUF...")
    model.save_pretrained(export_dir, safe_serialization=True)
    tokenizer.save_pretrained(export_dir)

    if os.path.exists(os.path.join(LOCAL_MODEL_DIR, 'tokenizer.model')):
        shutil.copy(os.path.join(LOCAL_MODEL_DIR, 'tokenizer.model'), os.path.join(export_dir, 'tokenizer.model'))

    gguf_path = f"/content/{export_name}.gguf"
    !python /content/llama.cpp/convert_hf_to_gguf.py {export_dir} --outfile {gguf_path}

    drive_path = f"/content/drive/MyDrive/quantization-results/{export_name}.gguf"
    if os.path.exists(gguf_path):
        shutil.move(gguf_path, drive_path)
        print(f"Successfully exported to Drive: {drive_path}")
    return drive_path

# Run for both models
export_to_gguf_proper("quantized/gemma_svd_60.pt", "gemma-4-E2B-svd60")
export_to_gguf_proper("quantized/gemma_svd_70.pt", "gemma-4-E2B-svd70")

Reconstructing quantized/gemma_svd_60.pt...
Injecting weights into 183 layers...
Saving HF model and converting to GGUF...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:hf-to-gguf:Loading model: gemma-2-2b-svd60
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: Gemma2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics detected bfloat16 tensor dtype, setting --outtype bf16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.bfloat16 --> BF16, shape = {2304, 256000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.bfloat16 --> F32, shape = {2304}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.bfloat16 --> BF16, shape = {9216, 2304}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.bfloat16 --> BF16, shape = {2304, 9216}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.bfloat16 --> BF16, shape = {2304, 9216}
INFO:hf-to-gguf:blk.0.post_attention_norm.weight,  torch.bfloat16 --> F32, shape = {2304}
INFO:hf-to-gguf:blk.0.post_f

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:hf-to-gguf:Loading model: gemma-2-2b-svd70
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: Gemma2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics detected bfloat16 tensor dtype, setting --outtype bf16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.bfloat16 --> BF16, shape = {2304, 256000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.bfloat16 --> F32, shape = {2304}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.bfloat16 --> BF16, shape = {9216, 2304}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.bfloat16 --> BF16, shape = {2304, 9216}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.bfloat16 --> BF16, shape = {2304, 9216}
INFO:hf-to-gguf:blk.0.post_attention_norm.weight,  torch.bfloat16 --> F32, shape = {2304}
INFO:hf-to-gguf:blk.0.post_f

'/content/drive/MyDrive/quantization-results/gemma-2-2b-svd70.gguf'

### **Step 2: Convert to GGUF using llama.cpp**
We will now install llama.cpp and convert our reconstructed fp16/bf16 weights into a GGUF file.

In [68]:
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -r /content/llama.cpp/requirements.txt -q

# Convert to GGUF (Unquantized first, then we can pack it)
# Output will be stored in /content/
!python /content/llama.cpp/convert_hf_to_gguf.py /content/gemma-2-svd-60-reconstructed --outfile /content/gemma-4-E2B-svd60.gguf

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 96757, done.
remote: Counting objects: 100% (219/219), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 96757 (delta 168), reused 118 (delta 118), pack-reused 96538 (from 3)
Receiving objects: 100% (96757/96757), 394.71 MiB | 37.18 MiB/s, done.
Resolving deltas: 100% (68959/68959), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 96.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

### **Step 3: Move to Google Drive**
Finally, we move the generated GGUF to your results folder in Drive.

In [69]:
import shutil

gguf_src = "/content/gemma-4-E2B-svd60.gguf"
drive_dest = "/content/drive/MyDrive/quantization-results/gemma-4-E2B-svd60.gguf"

if os.path.exists(gguf_src):
    shutil.move(gguf_src, drive_dest)
    print(f"Successfully exported GGUF to: {drive_dest}")
else:
    print("GGUF generation failed.")

Successfully exported GGUF to: /content/drive/MyDrive/quantization-results/gemma-2-2b-svd60.gguf


### **Step 4: Upload to Hugging Face Hub**
We will now upload the GGUF files and the reconstructed models to Hugging Face.

In [75]:
from huggingface_hub import HfApi, create_repo

# Configuration
hf_username = "toxzak" # Update this if your username is different
repo_id = f"{hf_username}/gemma-4-E2B-svd-quantized"

api = HfApi()

try:
    create_repo(repo_id=repo_id, token=token, repo_type="model", exist_ok=True)
    print(f"Repo {repo_id} is ready.")
except Exception as e:
    print(f"Error creating repo: {e}")

# Upload GGUF files from local content
gguf_files = ["gemma-4-E2B-svd60.gguf", "gemma-4-E2B-svd70.gguf"]
for file_name in gguf_files:
    local_file = f"/content/{file_name}"
    # Note: The previous step moved files to Drive, check local first then Drive
    if not os.path.exists(local_file):
        local_file = f"/content/drive/MyDrive/quantization-results/{file_name}"

    if os.path.exists(local_file):
        print(f"Uploading {file_name} to HF...")
        api.upload_file(
            path_or_fileobj=local_file,
            path_in_repo=file_name,
            repo_id=repo_id,
            token=token
        )
    else:
        print(f"Could not find {file_name} locally or in Drive.")

# Upload HF-format directories (folders containing safetensors)
hf_folders = ["gemma-4-E2B-svd60", "gemma-4-E2B-svd70"]
for folder in hf_folders:
    local_folder = f"/content/{folder}"
    if os.path.exists(local_folder):
        print(f"Uploading directory {folder} to HF...")
        api.upload_folder(
            folder_path=local_folder,
            repo_id=repo_id,
            path_in_repo=folder,
            token=token
        )

print(f"Done! View your models at: https://huggingface.co/{repo_id}")

Repo toxzak/gemma-2-2b-svd-quantized is ready.
Uploading gemma-2-2b-svd60.gguf to HF...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading gemma-2-2b-svd70.gguf to HF...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Uploading directory gemma-2-2b-svd70 to HF...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done! View your models at: https://huggingface.co/toxzak/gemma-2-2b-svd-quantized
